# NumPy y pandas · Práctica con NumPy

Continuación de *NumPy* con ejercicios cortos sobre funciones que aparecen a diario: matrices identidad, unir y partir arrays, operaciones de conjuntos, normalización, ordenación por columnas, borrar e insertar columnas, barajar con `default_rng`, relleno con `pad`, valores no finitos, frecuencias con `unique`, transpuestas, autovalores, distancias y ángulos entre vectores, y el coste en memoria de cada `dtype`. Ideas de [PYnative](https://pynative.com/python-numpy-exercise/) y de la hoja de ruta de [retosdeprogramacion.com](https://retosdeprogramacion.com/roadmap/).

In [1]:
import sys
from collections import Counter

import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. Matrices identidad

Crear la identidad de orden 4 con `np.eye` y con `np.identity` y comprobar que son iguales y que $AI = IA = A$ para una matriz cualquiera. Usar el parámetro `k` de `eye` para poner unos en la diagonal de encima y de debajo de la principal, y crear una identidad rectangular de 3 × 5.

In [2]:
I = np.eye(4, dtype=int)
print(I)
assert np.array_equal(I, np.identity(4, dtype=int))

A = np.arange(16).reshape(4, 4)
assert np.array_equal(A @ I, A) and np.array_equal(I @ A, A)

banda = np.eye(4, k=1, dtype=int) + np.eye(4, k=-1, dtype=int)
print("\nUnos junto a la diagonal:\n", banda)
print("\nRectangular 3 × 5:\n", np.eye(3, 5, dtype=int))
assert banda.sum() == 6 and np.trace(banda) == 0

[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]

Unos junto a la diagonal:
 [[0 1 0 0]
 [1 0 1 0]
 [0 1 0 1]
 [0 0 1 0]]

Rectangular 3 × 5:
 [[1 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]]


## 2. Unir y partir arrays

Con las ventas de dos semanas de tres tiendas, unir los arrays en horizontal y en vertical con `hstack`, `vstack` y `concatenate`. Después partir un vector de 10 elementos en 5 trozos iguales con `split`, ver que `split` en 3 falla y hacerlo con `array_split`, que admite trozos de distinto tamaño.

In [3]:
semana1 = np.array([[12, 15, 9], [20, 18, 22]])
semana2 = np.array([[11, 14, 10], [19, 21, 25]])

lado_a_lado = np.hstack([semana1, semana2])
una_encima = np.vstack([semana1, semana2])
print("hstack:\n", lado_a_lado)
print("vstack:\n", una_encima)
assert np.array_equal(lado_a_lado, np.concatenate([semana1, semana2], axis=1))
assert np.array_equal(una_encima, np.concatenate([semana1, semana2], axis=0))
assert lado_a_lado.shape == (2, 6) and una_encima.shape == (4, 3)

v = np.arange(10)
print("\nsplit en 5:", np.split(v, 5))
try:
    np.split(v, 3)
except ValueError as e:
    print("split en 3:", e)
trozos = np.array_split(v, 3)
print("array_split en 3:", trozos)
print("split por posiciones [2, 7]:", np.split(v, [2, 7]))
assert [len(t) for t in trozos] == [4, 3, 3]
assert np.array_equal(np.concatenate(trozos), v)

hstack:
 [[12 15  9 11 14 10]
 [20 18 22 19 21 25]]
vstack:
 [[12 15  9]
 [20 18 22]
 [11 14 10]
 [19 21 25]]

split en 5: [array([0, 1]), array([2, 3]), array([4, 5]), array([6, 7]), array([8, 9])]
split en 3: array split does not result in an equal division
array_split en 3: [array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]
split por posiciones [2, 7]: [array([0, 1]), array([2, 3, 4, 5, 6]), array([7, 8, 9])]


## 3. Operaciones de conjuntos

Dos grupos de senderismo tienen la lista de identificadores de sus socios. Calcular con NumPy quién está en los dos (`intersect1d`), en alguno (`union1d`), solo en el primero (`setdiff1d`) y en uno solo de los dos (`setxor1d`). Con `isin`, marcar qué socios de una lista de inscritos a una excursión pertenecen al primer grupo. Comprobarlo con conjuntos de Python.

In [4]:
grupo_norte = np.array([4, 8, 15, 16, 23, 42, 8])
grupo_sur = np.array([1, 4, 9, 16, 25, 36, 42])
inscritos = np.array([42, 7, 15, 30, 4])

operaciones = {
    "En los dos": np.intersect1d(grupo_norte, grupo_sur),
    "En alguno": np.union1d(grupo_norte, grupo_sur),
    "Solo en el norte": np.setdiff1d(grupo_norte, grupo_sur),
    "Solo en uno": np.setxor1d(grupo_norte, grupo_sur),
}
for nombre, resultado in operaciones.items():
    print(f"{nombre:<18}{resultado}")

mascara = np.isin(inscritos, grupo_norte)
print("\nInscritos del norte:", mascara, "→", inscritos[mascara])

n, s = set(grupo_norte.tolist()), set(grupo_sur.tolist())
for resultado, esperado in zip(operaciones.values(), (n & s, n | s, n - s, n ^ s)):
    assert resultado.tolist() == sorted(esperado)
assert inscritos[mascara].tolist() == [42, 15, 4]

En los dos        [ 4 16 42]
En alguno         [ 1  4  8  9 15 16 23 25 36 42]
Solo en el norte  [ 8 15 23]
Solo en uno       [ 1  8  9 15 23 25 36]

Inscritos del norte: [ True False  True False  True] → [42 15  4]


## 4. Normalizar columnas

Una matriz guarda la altura (cm), el peso (kg) y la edad de seis personas. Normalizar cada columna con min-máx, para que vaya de 0 a 1, y con z-score, para que tenga media 0 y desviación típica 1, sin bucles.

In [5]:
personas = np.array([
    [172, 70, 34],
    [160, 55, 28],
    [185, 90, 45],
    [168, 62, 52],
    [178, 81, 23],
    [155, 50, 61],
], dtype=float)

minimos, maximos = personas.min(axis=0), personas.max(axis=0)
min_max = (personas - minimos) / (maximos - minimos)
z = (personas - personas.mean(axis=0)) / personas.std(axis=0)
print("Min-máx:\n", min_max)
print("Z-score:\n", z)

assert np.allclose(min_max.min(axis=0), 0) and np.allclose(min_max.max(axis=0), 1)
assert np.allclose(z.mean(axis=0), 0) and np.allclose(z.std(axis=0), 1)
# El orden de los valores de cada columna no cambia al normalizar.
assert np.array_equal(np.argsort(z, axis=0), np.argsort(personas, axis=0))

Min-máx:
 [[0.567 0.5   0.289]
 [0.167 0.125 0.132]
 [1.    1.    0.579]
 [0.433 0.3   0.763]
 [0.767 0.775 0.   ]
 [0.    0.    1.   ]]
Z-score:
 [[ 0.229  0.142 -0.485]
 [-0.95  -0.925 -0.933]
 [ 1.507  1.565  0.336]
 [-0.164 -0.427  0.858]
 [ 0.819  0.925 -1.306]
 [-1.441 -1.28   1.53 ]]


## 5. Ordenar filas por una columna

Cada fila de una matriz es un alumno con su grupo, su nota y su edad. Ordenar las filas por nota con `argsort` (de menor a mayor y de mayor a menor) y después por grupo y, dentro de cada grupo, por nota descendente con `lexsort`, que toma como clave principal la última que recibe. Comprobarlo con `sorted`.

In [6]:
alumnos = np.array([
    [2, 7.5, 17],
    [1, 9.0, 16],
    [2, 5.0, 18],
    [1, 6.5, 17],
    [3, 8.0, 16],
    [1, 9.5, 17],
    [3, 4.5, 18],
])

por_nota = alumnos[alumnos[:, 1].argsort()]
por_nota_desc = alumnos[alumnos[:, 1].argsort()[::-1]]
por_grupo_y_nota = alumnos[np.lexsort((-alumnos[:, 1], alumnos[:, 0]))]
print("Por nota:\n", por_nota)
print("Por grupo y nota descendente:\n", por_grupo_y_nota)

filas = alumnos.tolist()
assert por_nota.tolist() == sorted(filas, key=lambda f: f[1])
assert por_nota_desc.tolist() == sorted(filas, key=lambda f: f[1], reverse=True)
assert por_grupo_y_nota.tolist() == sorted(filas, key=lambda f: (f[0], -f[1]))

Por nota:
 [[ 3.   4.5 18. ]
 [ 2.   5.  18. ]
 [ 1.   6.5 17. ]
 [ 2.   7.5 17. ]
 [ 3.   8.  16. ]
 [ 1.   9.  16. ]
 [ 1.   9.5 17. ]]
Por grupo y nota descendente:
 [[ 1.   9.5 17. ]
 [ 1.   9.  16. ]
 [ 1.   6.5 17. ]
 [ 2.   7.5 17. ]
 [ 2.   5.  18. ]
 [ 3.   8.  16. ]
 [ 3.   4.5 18. ]]


## 6. Borrar, insertar e intercambiar columnas

Partiendo de una matriz de 3 × 4, borrar la segunda columna con `np.delete`, insertar una columna de ceros en la posición 1 con `np.insert` y añadir otra al final con `np.column_stack`. Intercambiar la primera y la última columna con indexado avanzado. Comprobar que `delete` e `insert` devuelven copias y no cambian el original.

In [7]:
M = np.arange(1, 13).reshape(3, 4)
original = M.copy()

sin_segunda = np.delete(M, 1, axis=1)
con_ceros = np.insert(M, 1, 0, axis=1)
con_total = np.column_stack([M, M.sum(axis=1)])
print("Original:\n", M)
print("Sin la segunda columna:\n", sin_segunda)
print("Con ceros en la posición 1:\n", con_ceros)
print("Con el total de cada fila al final:\n", con_total)

intercambiada = M.copy()
intercambiada[:, [0, -1]] = intercambiada[:, [-1, 0]]
print("Primera y última intercambiadas:\n", intercambiada)

assert np.array_equal(M, original)
assert sin_segunda.shape == (3, 3) and np.array_equal(con_ceros[:, 1], [0, 0, 0])
assert np.array_equal(intercambiada[:, 0], M[:, -1]) and np.array_equal(intercambiada[:, -1], M[:, 0])
assert np.array_equal(intercambiada[:, 1:-1], M[:, 1:-1])

Original:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Sin la segunda columna:
 [[ 1  3  4]
 [ 5  7  8]
 [ 9 11 12]]
Con ceros en la posición 1:
 [[ 1  0  2  3  4]
 [ 5  0  6  7  8]
 [ 9  0 10 11 12]]
Con el total de cada fila al final:
 [[ 1  2  3  4 10]
 [ 5  6  7  8 26]
 [ 9 10 11 12 42]]
Primera y última intercambiadas:
 [[ 4  2  3  1]
 [ 8  6  7  5]
 [12 10 11  9]]


## 7. Ordenar por filas y barajar

Con un generador `np.random.default_rng(2026)`, crear una matriz de 4 × 5 enteros y ordenar cada fila (`np.sort(axis=1)`) y cada columna (`axis=0`). Barajar las filas en su sitio con `shuffle` y obtener una copia barajada con `permutation`, comprobando que las dos conservan las mismas filas.

In [8]:
rng = np.random.default_rng(2026)
datos = rng.integers(0, 100, size=(4, 5))
print("Datos:\n", datos)
print("Cada fila ordenada:\n", np.sort(datos, axis=1))
print("Cada columna ordenada:\n", np.sort(datos, axis=0))

barajada = rng.permutation(datos)
copia = datos.copy()
rng.shuffle(copia)
print("permutation (copia):\n", barajada)
print("shuffle (en su sitio):\n", copia)


def filas_ordenadas(m):
    return sorted(map(tuple, m.tolist()))


assert filas_ordenadas(barajada) == filas_ordenadas(copia) == filas_ordenadas(datos)
assert all(sorted(f) == g for f, g in zip(datos.tolist(), np.sort(datos, axis=1).tolist()))
assert np.array_equal(np.random.default_rng(2026).integers(0, 100, size=(4, 5)), datos)

Datos:
 [[85 17  2 63 36]
 [46  7 37 64 35]
 [83 79 70 90 72]
 [17 85 65  9 29]]
Cada fila ordenada:
 [[ 2 17 36 63 85]
 [ 7 35 37 46 64]
 [70 72 79 83 90]
 [ 9 17 29 65 85]]
Cada columna ordenada:
 [[17  7  2  9 29]
 [46 17 37 63 35]
 [83 79 65 64 36]
 [85 85 70 90 72]]
permutation (copia):
 [[85 17  2 63 36]
 [17 85 65  9 29]
 [83 79 70 90 72]
 [46  7 37 64 35]]
shuffle (en su sitio):
 [[46  7 37 64 35]
 [83 79 70 90 72]
 [17 85 65  9 29]
 [85 17  2 63 36]]


## 8. Borde de unos

Crear una matriz de 5 × 6 con unos en el borde y ceros dentro de dos formas: rellenando una matriz de ceros con `np.pad` y poniendo a cero el interior de una matriz de unos con cortes. Añadir también un marco de dos capas con valores distintos a una matriz de 2 × 2.

In [9]:
con_pad = np.pad(np.zeros((3, 4), dtype=int), pad_width=1, constant_values=1)
con_cortes = np.ones((5, 6), dtype=int)
con_cortes[1:-1, 1:-1] = 0
print(con_pad)
assert np.array_equal(con_pad, con_cortes)

centro = np.array([[5, 5], [5, 5]])
marco = np.pad(np.pad(centro, 1, constant_values=2), 1, constant_values=1)
print("\nMarco de dos capas:\n", marco)
assert marco.shape == (6, 6) and marco.sum() == 4 * 5 + 12 * 2 + 20 * 1

[[1 1 1 1 1 1]
 [1 0 0 0 0 1]
 [1 0 0 0 0 1]
 [1 0 0 0 0 1]
 [1 1 1 1 1 1]]

Marco de dos capas:
 [[1 1 1 1 1 1]
 [1 2 2 2 2 1]
 [1 2 5 5 2 1]
 [1 2 5 5 2 1]
 [1 2 2 2 2 1]
 [1 1 1 1 1 1]]


## 9. Tablero de ajedrez

Crear un tablero de 8 × 8 con 1 en las casillas negras y 0 en las blancas de tres formas: con cortes de paso 2, con `np.tile` y con la paridad de la suma de fila y columna. En el tablero real la casilla a1 (abajo a la izquierda) es negra. Mostrarlo con caracteres.

In [10]:
cortes = np.zeros((8, 8), dtype=int)
cortes[::2, ::2] = 1
cortes[1::2, 1::2] = 1

mosaico = np.tile([[1, 0], [0, 1]], (4, 4))
filas_idx, columnas_idx = np.indices((8, 8))
paridad = ((filas_idx + columnas_idx) % 2 == 0).astype(int)

assert np.array_equal(cortes, mosaico) and np.array_equal(cortes, paridad)

# La fila 0 del array es la fila 8 del tablero: se muestra al revés para que a1 quede abajo.
tablero = cortes[::-1]
for numero, fila in zip(range(8, 0, -1), tablero):
    print(numero, " ".join("■" if c else "□" for c in fila))
print("  " + " ".join("abcdefgh"))
assert tablero[-1, 0] == 1 and tablero.sum() == 32

8 □ ■ □ ■ □ ■ □ ■
7 ■ □ ■ □ ■ □ ■ □
6 □ ■ □ ■ □ ■ □ ■
5 ■ □ ■ □ ■ □ ■ □
4 □ ■ □ ■ □ ■ □ ■
3 ■ □ ■ □ ■ □ ■ □
2 □ ■ □ ■ □ ■ □ ■
1 ■ □ ■ □ ■ □ ■ □
  a b c d e f g h


## 10. Valores no finitos

Una matriz de lecturas de tres sensores tiene huecos (`nan`) y una lectura infinita por un fallo. Contar con `isnan` e `isfinite` los valores ausentes y los no finitos, y sustituir cada valor no finito por la media de los valores válidos de su columna (`nanmean`).

In [11]:
lecturas = np.array([
    [21.5, 60.0, np.nan],
    [22.0, np.nan, 1013.0],
    [np.inf, 58.0, 1011.0],
    [21.0, 62.0, 1012.0],
    [np.nan, 61.0, 1010.0],
])

print("nan por columna:       ", np.isnan(lecturas).sum(axis=0))
print("no finitos por columna:", (~np.isfinite(lecturas)).sum(axis=0))

limpias = np.where(np.isfinite(lecturas), lecturas, np.nan)
medias = np.nanmean(limpias, axis=0)
rellenas = np.where(np.isnan(limpias), medias, limpias)
print("Medias de los válidos:", medias)
print("Rellenas:\n", rellenas)

assert np.isfinite(rellenas).all()
for j in range(lecturas.shape[1]):
    validos = [x for x in lecturas[:, j] if np.isfinite(x)]
    assert np.isclose(medias[j], sum(validos) / len(validos))

nan por columna:        [1 1 1]
no finitos por columna: [2 1 1]
Medias de los válidos: [  21.5    60.25 1011.5 ]
Rellenas:
 [[  21.5    60.   1011.5 ]
 [  22.     60.25 1013.  ]
 [  21.5    58.   1011.  ]
 [  21.     62.   1012.  ]
 [  21.5    61.   1010.  ]]


## 11. Valores únicos y frecuencias

Simular 60 tiradas de un dado con una semilla fija y obtener con `np.unique` los valores distintos, cuántas veces sale cada uno (`return_counts`) y el valor más frecuente. Usar `return_inverse` para reconstruir las tiradas a partir de los únicos. Comprobarlo con `Counter`.

In [12]:
tiradas = np.random.default_rng(7).integers(1, 7, size=60)
valores, veces = np.unique(tiradas, return_counts=True)
for valor, n in zip(valores, veces):
    print(f"{valor}: {n:>2} {'●' * n}")
print("Más frecuente:", valores[veces.argmax()])

_, inverso = np.unique(tiradas, return_inverse=True)
assert np.array_equal(valores[inverso], tiradas)
assert dict(zip(valores.tolist(), veces.tolist())) == Counter(tiradas.tolist())
assert veces.sum() == 60

1:  9 ●●●●●●●●●
2:  7 ●●●●●●●
3: 10 ●●●●●●●●●●
4: 11 ●●●●●●●●●●●
5: 11 ●●●●●●●●●●●
6: 12 ●●●●●●●●●●●●
Más frecuente: 6


## 12. Transpuesta y sus propiedades

Comprobar con matrices aleatorias de enteros que $(A^T)^T = A$, $(A + B)^T = A^T + B^T$, $(AB)^T = B^T A^T$ y $\det(A^T) = \det(A)$, y que $A + A^T$ es simétrica. Ver que `A.T` no copia los datos: es una vista sobre el mismo bloque de memoria.

In [13]:
rng = np.random.default_rng(3)
A = rng.integers(-5, 6, size=(3, 3))
B = rng.integers(-5, 6, size=(3, 3))
print("A:\n", A)
print("A.T:\n", A.T)

assert np.array_equal(A.T.T, A)
assert np.array_equal((A + B).T, A.T + B.T)
assert np.array_equal((A @ B).T, B.T @ A.T)
assert np.isclose(np.linalg.det(A.T), np.linalg.det(A))
S = A + A.T
assert np.array_equal(S, S.T)

print("¿A.T comparte memoria con A?", np.shares_memory(A, A.T))
vista = A.T
vista[0, 1] = 99
print("Tras cambiar A.T[0, 1], A[1, 0] =", A[1, 0])
assert A[1, 0] == 99

R = rng.integers(0, 10, size=(2, 4))
assert R.T.shape == (4, 2) and R.T[3, 1] == R[1, 3]

A:
 [[ 3 -5 -4]
 [-3 -4  3]
 [ 4  1 -5]]
A.T:
 [[ 3 -3  4]
 [-5 -4  1]
 [-4  3 -5]]
¿A.T comparte memoria con A? True
Tras cambiar A.T[0, 1], A[1, 0] = 99


## 13. Autovalores y autovectores

Calcular con `np.linalg.eig` los autovalores y autovectores de $A = \begin{pmatrix} 4 & 1 \\ 2 & 3 \end{pmatrix}$ y comprobar que $A v = \lambda v$ para cada par. Comprobar también que la suma de los autovalores es la traza y su producto el determinante. Con una matriz simétrica, usar `eigh` y ver que los autovectores son ortogonales.

In [14]:
A = np.array([[4, 1], [2, 3]])
autovalores, autovectores = np.linalg.eig(A)
# eig puede devolver complejos con parte imaginaria nula; se pasan a reales si lo son.
autovalores, autovectores = np.real_if_close(autovalores), np.real_if_close(autovectores)
for i, lam in enumerate(autovalores):
    v = autovectores[:, i]
    print(f"λ = {lam:.1f}, v = {v}, A·v = {A @ v}, λ·v = {lam * v}")
    assert np.allclose(A @ v, lam * v)

assert np.allclose(sorted(autovalores), [2, 5])
assert np.isclose(autovalores.sum(), np.trace(A))
assert np.isclose(autovalores.prod(), np.linalg.det(A))
assert np.allclose(A @ autovectores, autovectores * autovalores)

S = np.array([[2, 1, 0], [1, 2, 1], [0, 1, 2]], dtype=float)
valores_s, vectores_s = np.linalg.eigh(S)
print("\nAutovalores de la simétrica:", valores_s)
print("VᵀV =\n", vectores_s.T @ vectores_s)
assert np.allclose(vectores_s.T @ vectores_s, np.eye(3))
assert np.allclose(valores_s, [2 - np.sqrt(2), 2, 2 + np.sqrt(2)])

λ = 5.0, v = [0.707 0.707], A·v = [3.536 3.536], λ·v = [3.536 3.536]
λ = 2.0, v = [-0.447  0.894], A·v = [-0.894  1.789], λ·v = [-0.894  1.789]

Autovalores de la simétrica: [0.586 2.    3.414]
VᵀV =
 [[ 1. -0. -0.]
 [-0.  1.  0.]
 [-0.  0.  1.]]


## 14. Elemento más cercano a un valor

Encontrar en un array de alturas de picos canarios (en metros) el más cercano a un valor dado con `argmin` sobre la diferencia absoluta. Hacerlo después para varios valores a la vez con broadcasting y comprobarlo con `min` y una clave.

In [15]:
picos = {"Teide": 3715, "Roque de los Muchachos": 2426, "Pico de las Nieves": 1949,
         "Garajonay": 1487, "Malpaso": 1501, "Peñas del Chache": 671, "Pico de la Zarza": 807}
nombres = np.array(list(picos))
alturas = np.array(list(picos.values()))


def mas_cercano(valores, objetivo):
    return np.abs(valores - objetivo).argmin()


i = mas_cercano(alturas, 1500)
print(f"Más cercano a 1500 m: {nombres[i]} ({alturas[i]} m)")

objetivos = np.array([700, 2000, 3000, 1490])
indices = np.abs(alturas[:, None] - objetivos).argmin(axis=0)
for objetivo, j in zip(objetivos, indices):
    print(f"{objetivo:>5} m → {nombres[j]} ({alturas[j]} m)")
    assert alturas[j] == min(picos.values(), key=lambda h: abs(h - objetivo))
assert nombres[i] == "Malpaso"

Más cercano a 1500 m: Malpaso (1501 m)
  700 m → Peñas del Chache (671 m)
 2000 m → Pico de las Nieves (1949 m)
 3000 m → Roque de los Muchachos (2426 m)
 1490 m → Garajonay (1487 m)


## 15. Memoria según el `dtype`

Crear un array de un millón de elementos con distintos `dtype` y mostrar `itemsize` (bytes por elemento) y `nbytes` (bytes totales). Ver qué se pierde al ahorrar memoria: los enteros de 8 bits desbordan y `float16` pierde precisión.

In [16]:
tipos = [np.bool_, np.int8, np.int16, np.int32, np.int64, np.float16, np.float32, np.float64, np.complex128]
print(f"{'dtype':<12}{'itemsize':>9}{'nbytes':>12}{'MB':>8}")
for tipo in tipos:
    a = np.zeros(1_000_000, dtype=tipo)
    print(f"{a.dtype.name:<12}{a.itemsize:>9}{a.nbytes:>12}{a.nbytes / 2**20:>8.2f}")
    assert a.nbytes == a.size * a.itemsize

print("\nint8: 100 + 100 =", np.array([100], dtype=np.int8) + np.array([100], dtype=np.int8))
print("Rango de int8:", np.iinfo(np.int8).min, "a", np.iinfo(np.int8).max)
print(f"float16(2049) = {float(np.float16(2049)):.0f} | float32(0.1) = {np.float32(0.1):.10f}")
assert (np.array([100], dtype=np.int8) * 2)[0] == -56
assert np.float16(2049) == 2048

dtype        itemsize      nbytes      MB
bool                1     1000000    0.95
int8                1     1000000    0.95
int16               2     2000000    1.91
int32               4     4000000    3.81
int64               8     8000000    7.63
float16             2     2000000    1.91
float32             4     4000000    3.81
float64             8     8000000    7.63
complex128         16    16000000   15.26

int8: 100 + 100 = [-56]
Rango de int8: -128 a 127
float16(2049) = 2048 | float32(0.1) = 0.1000000015


## 16. Producto escalar, ortogonalidad y ángulos

Calcular el producto escalar de dos vectores con `np.dot`, `@` y a mano. Escribir funciones que digan si dos vectores son ortogonales y que calculen el ángulo entre ellos en grados, $\theta = \arccos\frac{u \cdot v}{\lVert u \rVert \lVert v \rVert}$, probándolas con pares cuyo ángulo se conoce.

In [17]:
def son_ortogonales(u, v):
    return np.isclose(np.dot(u, v), 0)


def angulo(u, v):
    coseno = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    # Los redondeos pueden dar 1.0000000002 y arccos devolvería nan.
    return np.degrees(np.arccos(np.clip(coseno, -1, 1)))


u, v = np.array([1, 2, 2]), np.array([2, 1, -2])
print("u·v =", np.dot(u, v), "=", u @ v, "=", sum(a * b for a, b in zip(u, v)))
print("¿Ortogonales?", son_ortogonales(u, v))

pares = [((1, 0), (1, 1), 45), ((1, 0), (0, 3), 90), ((2, 2), (-1, -1), 180),
         ((1, 0), (1, np.sqrt(3)), 60), ((3, 4), (6, 8), 0)]
for a, b, esperado in pares:
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    print(f"{str(a):<14}{str(b):<14}{angulo(a, b):>7.2f}°")
    assert np.isclose(angulo(a, b), esperado)
assert son_ortogonales(u, v) and not son_ortogonales(np.array([1, 1]), np.array([1, 0]))

u·v = 0 = 0 = 0
¿Ortogonales? True
[1. 0.]       [1. 1.]         45.00°
[1. 0.]       [0. 3.]         90.00°
[2. 2.]       [-1. -1.]      180.00°
[1. 0.]       [1.    1.732]   60.00°
[3. 4.]       [6. 8.]          0.00°


## 17. Arrays de objetos frente a arrays numéricos

Comparar un array con `dtype=object` con uno numérico: ver qué tipo deduce NumPy de una lista que mezcla números y texto, que con `object` cada elemento es un objeto de Python (enteros sin límite, pero operaciones elemento a elemento en Python) y que con `int64` los enteros grandes desbordan sin avisar. Comparar cuánto ocupa cada elemento en los dos casos y comprobar también que con `object` no funcionan las funciones que necesitan un tipo numérico, como `np.sqrt`.

In [18]:
mezcla = np.array([1, "dos", 3.0])
print("Lista mixta sin dtype:", mezcla, "→", mezcla.dtype)
assert mezcla.dtype.kind == "U" and mezcla[0] == "1"

objetos = np.array([1, "dos", 3.0, None], dtype=object)
print("Con dtype=object:", objetos, "→ tipos", [type(x).__name__ for x in objetos])

grandes_obj = np.array([2**62, 2**70], dtype=object)
grandes_int = np.array([2**62], dtype=np.int64)
print("\nobject × 4:", grandes_obj * 4)
print("int64 × 4: ", grandes_int * 4)
assert grandes_obj[1] * 4 == 2**72 and (grandes_int * 4)[0] == 0

numerico = np.arange(1, 6)
como_objeto = numerico.astype(object)
print("\nSuma igual con los dos:", numerico.sum(), como_objeto.sum())
por_objeto = sys.getsizeof(como_objeto[0])
print(f"Bytes por elemento: int64 {numerico.itemsize}; object {como_objeto.itemsize} del puntero"
      f" + {por_objeto} del int de Python al que apunta")
assert type(como_objeto[0]) is int and por_objeto > numerico.itemsize
try:
    np.sqrt(como_objeto)
except TypeError as e:
    print("np.sqrt sobre object:", e)
assert numerico.sum() == como_objeto.sum() == 15

Lista mixta sin dtype: ['1' 'dos' '3.0'] → <U32
Con dtype=object: [1 'dos' 3.0 None] → tipos ['int', 'str', 'float', 'NoneType']

object × 4: [18446744073709551616 4722366482869645213696]
int64 × 4:  [0]

Suma igual con los dos: 15 15
Bytes por elemento: int64 8; object 8 del puntero + 28 del int de Python al que apunta
np.sqrt sobre object: loop of ufunc does not support argument 0 of type int which has no callable sqrt method
